# Module 03 — Structured output

**THE ONE IDEA:** parsing free text is a *bug*, not a chore. Constrained output
deletes an entire failure class instead of catching it.

Three attempts at the same job, each stronger than the last:

| | Guarantee |
|---|---|
| 1. Free text + regex | none — you are guessing |
| 2. JSON mode | valid JSON, but *any* shape |
| 3. `strict` JSON schema | valid JSON **in your shape** |

Attempt 3 is why module 13 can validate tool arguments at all.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
from _providers import get_client
import json
from pydantic import BaseModel, ValidationError

client, MODEL, _ = get_client("openai")

APPLICATION = ("Applicant Sarah Chen, earns 74000 a year, wants to borrow 285000 "
               "on a property valued at 320000. First-time buyer. 5-year fix.")

class Loan(BaseModel):
    applicant: str
    income: float
    amount: float
    property_value: float
    first_time_buyer: bool

print("target schema:", list(Loan.model_fields))

## Attempt 1 — ask nicely, then parse

The prompt says "return JSON". Nothing enforces it. The model may add a preamble,
wrap it in a code fence, or use different key names on a different day.

In [ ]:
r = client.chat.completions.create(
    model=MODEL, max_tokens=300,
    messages=[{"role": "user",
               "content": f"Extract the loan details as JSON.\n\n{APPLICATION}"}],
)
raw = r.choices[0].message.content
print("RAW:\n", raw[:260])

try:
    Loan.model_validate_json(raw)
    print("\nparsed -> worked THIS time. Nothing guaranteed it would.")
except (json.JSONDecodeError, ValidationError) as e:
    print("\nparsed -> FAILED:", str(e)[:150])

## Attempt 2 — JSON mode

`response_format={"type": "json_object"}` guarantees the output *parses*.
It does **not** guarantee your field names, your types, or that nothing is missing.

In [ ]:
r = client.chat.completions.create(
    model=MODEL, max_tokens=300,
    response_format={"type": "json_object"},
    messages=[{"role": "user",
               "content": f"Extract the loan details as JSON.\n\n{APPLICATION}"}],
)
raw = r.choices[0].message.content
print("RAW:", raw[:200])
print("\njson.loads ->", "OK" if json.loads(raw) else "?")
try:
    Loan.model_validate_json(raw); print("matches MY schema -> yes (lucky)")
except ValidationError as e:
    print("matches MY schema -> NO:", str(e).splitlines()[0])

## Attempt 3 — strict JSON schema

Now the shape is part of the request. The model is constrained at decode time, so an
off-schema response is not merely unlikely — it is unrepresentable.

In [ ]:
schema = Loan.model_json_schema()
schema["additionalProperties"] = False          # required by strict mode

r = client.chat.completions.create(
    model=MODEL, max_tokens=300,
    response_format={"type": "json_schema",
                     "json_schema": {"name": "loan", "strict": True, "schema": schema}},
    messages=[{"role": "user",
               "content": f"Extract the loan details.\n\n{APPLICATION}"}],
)
loan = Loan.model_validate_json(r.choices[0].message.content)
print(loan.model_dump_json(indent=2))

ltv = loan.amount / loan.property_value * 100
print(f"\nLTV = {ltv:.1f}%   (arithmetic on a TYPED field, not on a guess)")

## The lesson

In [ ]:
print("LESSON — free-text parsing is a bug you re-discover in production.")
print()
print("  attempt 1  regex/json on prose   -> fails on a preamble, a fence, a rename")
print("  attempt 2  json mode             -> always parses, shape still unknown")
print("  attempt 3  strict json schema    -> shape guaranteed, class deleted")
print()
print("You did not ADD a check in attempt 3. You removed the need for one — the")
print("decoder cannot emit an off-schema token. That is the difference between")
print("catching a failure and making it impossible.")
print()
print("Module 13 reuses exactly this to validate TOOL ARGUMENTS, which is where a")
print("hallucinated field name stops being cosmetic and starts calling the wrong API.")

---

**Next:** `../B_workflows/04_workflow_prompt_chaining.ipynb`